# 04. Preprocesamiento de Features de Contenido

Este notebook transforma el enriquecimiento IMDb + MovieLens en un conjunto de features reutilizable para híbrido y deep learning.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

from collections import Counter

from src.utils.recsys_utils import build_movies_enriched, ensure_dir, flatten_column, load_clean_data, prepare_content_features

imdb, movielens = load_clean_data(source="both", use_cache=True)
processed_dir = ensure_dir(ROOT / 'data' / 'processed')
tables_dir = ensure_dir(ROOT / 'reports' / 'tables')
items_master = build_movies_enriched(imdb, movielens, use_cache=True)


[20:52:14] INFO - Iniciando pipeline de carga y limpieza | source=both | use_cache=True | refresh_cache=False
[20:52:14] INFO - Cargando cache limpia para imdb
[20:52:44] INFO - Cache cargada: name.basics.pkl | filas=15257687 columnas=8
[20:55:11] INFO - Cache cargada: title.akas.pkl | filas=55865762 columnas=10
[20:55:41] INFO - Cache cargada: title.basics.pkl | filas=12446644 columnas=10
[20:55:48] INFO - Cache cargada: title.crew.pkl | filas=12446644 columnas=5
[20:55:49] INFO - Cache cargada: title.episode.pkl | filas=9612089 columnas=4
[20:56:05] INFO - Cache cargada: title.principals.pkl | filas=99029989 columnas=6
[20:56:05] INFO - Cache cargada: title.ratings.pkl | filas=1662797 columnas=3
[20:56:05] INFO - Cargando cache limpia para movielens
[20:56:05] INFO - Cache cargada: links.pkl | filas=87585 columnas=4
[20:56:05] INFO - Cache cargada: movies.pkl | filas=87585 columnas=6
[20:56:05] INFO - Cache cargada: ratings.pkl | filas=32000204 columnas=4
[20:56:05] INFO - Cache carg

## Construir tabla maestra de ítems

Se reutiliza el join del notebook base y se agregan señales de personas para reducir el trabajo repetido en modelado.


In [2]:
item_quality = pd.DataFrame([
    {'metric': 'items_master_rows', 'value': len(items_master)},
    {'metric': 'with_directors', 'value': items_master['directors_list'].apply(len).gt(0).mean()},
    {'metric': 'with_writers', 'value': items_master['writers_list'].apply(len).gt(0).mean()},
    {'metric': 'with_principals', 'value': items_master['principal_names'].apply(len).gt(0).mean()},
])
display(item_quality)

,metric,value
0,items_master_rows,87585.000000
1,with_directors,0.993001
2,with_writers,0.931301
3,with_principals,0.996483


## Crear features estructuradas

Se generan variables numéricas, one-hot de `titleType` y bolsas controladas para directores y cast principal.


In [3]:
item_features = prepare_content_features(items_master, movielens['tags'])
item_features = item_features.merge(
    pd.get_dummies(items_master[['movieId', 'titleType']], columns=['titleType'], dummy_na=True),
    on='movieId',
    how='left'
)

def top_token_frame(series, prefix, top_k=50):
    token_counts = Counter(flatten_column(series))
    top_tokens = {token for token, _ in token_counts.most_common(top_k)}
    rows = []
    for values in series:
        current = set(values if isinstance(values, list) else [])
        rows.append({f'{prefix}_{token}': int(token in current) for token in top_tokens})
    return pd.DataFrame(rows)

director_features = top_token_frame(items_master['directors_list'], 'director', top_k=40)
principal_features = top_token_frame(items_master['principal_names'], 'principal', top_k=60)

item_features = pd.concat([
    item_features.reset_index(drop=True),
    director_features.reset_index(drop=True),
    principal_features.reset_index(drop=True),
], axis=1)

display(item_features.head())
print('Shape final de item_features:', item_features.shape)


[20:57:04] INFO - Preparando features de contenido
[20:57:04] INFO - Construyendo matrices de generos
[20:57:04] INFO - Construyendo features TF-IDF de tags
[20:57:13] INFO - Convirtiendo columnas sparse a densas | n_cols=500
[20:57:13] INFO - Features de contenido listas | filas=87585 columnas=508


,movieId,title,clean_title,release_year,startYear,runtimeMinutes,averageRating,numVotes,tag_tfidf_11,tag_tfidf_1980s,...,titleType_short,titleType_tvEpisode,titleType_tvMiniSeries,titleType_tvMovie,titleType_tvSeries,titleType_tvShort,titleType_tvSpecial,titleType_video,titleType_videoGame,titleType_nan
0,1,Toy Story (1995),Toy Story,1995,1995,81,8.3,1171117,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False
1,2,Jumanji (1995),Jumanji,1995,1995,104,7.1,412163,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False
2,3,Grumpier Old Men (1995),Grumpier Old Men,1995,1995,101,6.7,31652,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False
3,4,Waiting to Exhale (1995),Waiting to Exhale,1995,1995,124,6.0,13798,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False
4,5,Father of the Bride Part II (1995),Father of the Bride Part II,1995,1995,106,6.1,45070,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False


Shape final de item_features: (87585, 519)


## Guardar artefactos de contenido

Se dejan listos `items_master` y `item_features` para los notebooks de híbrido y deep learning.


In [4]:
items_master.to_pickle(processed_dir / 'items_master.pkl')
item_features.to_parquet(processed_dir / 'item_features.parquet', index=False)
item_quality.to_csv(tables_dir / 'item_feature_quality.csv', index=False)
print('Guardado items_master.pkl y item_features.parquet')


Guardado items_master.pkl y item_features.parquet
